In [ ]:
import numpy as np
import pandas as pd

import diffuse
import focus
import os
import cv2

from matplotlib import pyplot as plt
from tqdm import tqdm

from diffuse import img_dir, mask_dir

In [ ]:

testim = diffuse.get_img('giant_panda_o0_n1.jpg', img_dir=img_dir)[...,::-1]
testmask = diffuse.get_mask('giant_panda_o0_n1.jpg', mask_dir=mask_dir)

def write(im, msk, sv):
    cv2.imwrite(sv, im)#[...,::-1])
    cv2.imwrite(sv.replace('imgs', 'masks'), msk)
    
def test_output(func):
    return func(testim, testmask)

def display_output(rm, flip=False):
    fig, ax = plt.subplots(1,2, figsize=[1.5,1])
    for i, (a, r) in enumerate(zip(ax, rm)):
        if flip and i==0: a.imshow(r[:,:,::-1]); a.axis('off')
        else: a.imshow(r); a.axis('off')
    fig.suptitle('occlusion: %.3f' % diffuse.calculate_occlusion_w_mask(testmask, rm[-1]), size='x-small', y=0.8)
    fig.tight_layout()

In [ ]:
rm = test_output(diffuse.random_mask)
display_output(rm)

In [ ]:
rm = test_output(diffuse.gate_occlusion)
display_output(rm)

In [ ]:
rm = test_output(diffuse.xhatch_occlusion)
display_output(rm)

In [ ]:
rm = test_output(diffuse.rotated_gate_occlusion)
rm = diffuse.rotated_gate_occlusion(*rm, angle=-45)
display_output(rm)

In [ ]:
#testim = testim[:,:,::-1]
rm = diffuse.gate_occlusion(testim, testmask, mode='vertical')
display_output(rm)

In [ ]:
testim = testim[:,:,::-1]
rm = diffuse.object_occlude(testim, testmask, diffuse.read_object('leaf.png'), angle=45, scale=0.1)
for _ in range(8):
    rm = diffuse.object_occlude(*rm, diffuse.read_object('leaf%s.png' % np.random.choice(['', '_dark', '_yellow', '_orange'])), angle=np.random.randint(-180,180), scale=0.1)
display_output(rm, flip=True)

In [ ]:
'''clear_0 = focus.annos(0)
focus.focus(clear_0)
clear_0 = focus.filter_clear(clear_0)
clear_0.drop(columns=['px', 'blur']).to_csv('clear_0.csv', sep=' ', header=False, index=False)'''

In [ ]:
def exists_or_make(x): 
    if not os.path.exists(x): os.mkdir(x)

exists_or_make('gen_imgs')
exists_or_make('gen_masks')

for x in [os.path.join('gen_imgs', x) for x in ['vgates', 'hgates', 'xgates', 'rotgates', 'leaves', 'boxes']]: 
    exists_or_make(x)

for x in [os.path.join('gen_masks', x) for x in ['vgates', 'hgates', 'xgates', 'rotgates', 'leaves', 'boxes']]: 
    exists_or_make(x)

In [ ]:
'''box_anno = []
vga_anno = []
hga_anno = []
xga_anno = []
rga_anno = []
lea_anno = []

for rec in tqdm(clear_0.iloc, desc='Generating images', total=len(clear_0)):
    f = rec.fname
    c = rec.cls

    try:
        imin = diffuse.get_img(focus.floc(f))
        if os.path.exists(focus.mloc(f)):
            mskin = diffuse.get_mask(focus.mloc(f))
        else: continue
    except:
        continue

    if np.sum(mskin) < 1: continue

    # boxes
    occ = 0
    iters = 0
    while occ <= 10 and iters < 50:
        imout, mout = diffuse.random_mask(imin, mskin)
        occ = int(100*diffuse.calculate_occlusion_w_mask(mskin, mout))
        iters += 1
    sv = 'gen_imgs/boxes/occ%i_%s' % (occ, f.split(os.sep)[-1])
    box_anno.append({'fname': sv, 'cls': c})
    write(imout, mout, sv)

    # vgates
    width, spacing = np.random.choice([2, 5, 9]), np.random.choice([10, 15, 20])
    imout, mout = diffuse.gate_occlusion(imin, mskin, width=width, spacing=spacing, mode='vertical')
    occ = int(100*diffuse.calculate_occlusion_w_mask(mskin, mout))
    sv = 'gen_imgs/vgates/occ%i_%s' % (occ, f.split(os.sep)[-1])
    vga_anno.append({'fname': sv, 'cls': c})
    write(imout, mout, sv)

    # hgates
    width, spacing = np.random.choice([2, 5, 9]), np.random.choice([10, 15, 20])
    imout, mout = diffuse.gate_occlusion(imin, mskin, width=width, spacing=spacing, mode='horizontal')
    occ = int(100*diffuse.calculate_occlusion_w_mask(mskin, mout))
    sv = 'gen_imgs/hgates/occ%i_%s' % (occ, f.split(os.sep)[-1])
    hga_anno.append({'fname': sv, 'cls': c})
    write(imout, mout, sv)

    # xgates
    width, spacing = np.random.choice([2, 5, 9]), np.random.choice([10, 15, 20])
    imout, mout = diffuse.xhatch_occlusion(imin, mskin, width=width, spacing=spacing)
    occ = int(100*diffuse.calculate_occlusion_w_mask(mskin, mout))
    sv = 'gen_imgs/xgates/occ%i_%s' % (occ, f.split(os.sep)[-1])
    xga_anno.append({'fname': sv, 'cls': c})
    write(imout, mout, sv)

    # rotgates
    width, spacing = np.random.choice([2, 5, 9]), np.random.choice([10, 15, 20])
    angle_pairs = [(-45, 45), (45,-45), (30,-30), (-30, 30), (60,-60), (-60,60), (30,-60), (-30,60), (60,-30), (-60,30)]
    angle_pair = angle_pairs[np.random.randint(0,len(angle_pairs))]
    imout, mout = diffuse.rotated_gate_occlusion(imin, mskin, angle=angle_pair[0], width=width, spacing=spacing)
    if np.random.random() > 0.5: imout, mout = diffuse.rotated_gate_occlusion(imout, mout, angle=angle_pair[1], width=width, spacing=spacing)
    occ = int(100*diffuse.calculate_occlusion_w_mask(mskin, mout))
    sv = 'gen_imgs/rotgates/occ%i_%s' % (occ, f.split(os.sep)[-1])
    rga_anno.append({'fname': sv, 'cls': c})
    write(imout, mout, sv)

    # leaves
    rand_occ = np.random.randint(10,90)
    iters = 0
    imout, mout = diffuse.object_occlude(imin, mskin, diffuse.read_object('leaf%s.png' % np.random.choice(['', '_dark', '_orange', '_yellow'])), flip=False, angle=np.random.randint(-180,180), scale=0.02+0.12*np.random.random())
    occ = int(100*diffuse.calculate_occlusion_w_mask(mskin, mout))
    while occ < rand_occ and iters < 50:
        imout, mout = diffuse.object_occlude(imout, mout, diffuse.read_object('leaf%s.png' % np.random.choice(['', '_dark', '_orange', '_yellow'])), flip=False, angle=np.random.randint(-180,180), scale=0.02+0.12*np.random.random())
        occ = int(100*diffuse.calculate_occlusion_w_mask(mskin, mout))
        iters += 1
    sv = 'gen_imgs/leaves/occ%i_%s' % (occ, f.split(os.sep)[-1])
    lea_anno.append({'fname': sv, 'cls': c})
    write(imout, mout, sv)'''

In [ ]:
'''exists_or_make('csvs')

for ana, afl in zip(['box', 'vga', 'hga', 'xga', 'rga', 'lea'], [box_anno, vga_anno, hga_anno, xga_anno, rga_anno, lea_anno]):
    afl = pd.DataFrame.from_records(afl)
    afl.to_csv('csvs/%s.csv' % ana, sep=' ', header=False, index=False)'''

In [ ]:
'''anno_files = []
for anno_file in ['csvs/%s.csv' % a for a in ['box', 'hga', 'lea', 'rga', 'vga', 'xga']]:
    if os.path.exists(anno_file):
        anno_files.append(pd.read_csv(anno_file, sep=' ', names=['fname', 'cls']))
if len(anno_files) > 0:
    anno_files = pd.concat(anno_files, axis=0)
    anno_files.to_csv('diffuse.csv', sep=' ', header=False, index=False)'''